# 🐝 Lab 3.3 — Beyond Dense Attention: Mixture-of-Experts & State-Space Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab09_moe_and_ssm.ipynb)

**⏱ Time:** ~2 hours  ·  **Day 3 — Transformers & Beyond**

A dense Transformer makes *every* parameter work on *every* token, and makes *every* token look at *every* other token. Two ideas break those two "everything" costs:

* **Mixture-of-Experts (MoE)** keeps many MLPs ("experts") but routes each token to only a few: more parameters, the same FLOPs per token. Like a hive where each bee only visits the flowers it's good at. 🌼
* **State-space models (SSMs)** like Mamba replace attention's growing memory with a fixed-size recurrent state that's updated in linear time, and can be *trained* in parallel with a scan.

### 🎯 Learning objectives
- Implement a **top-k MoE layer** with expert **capacity** and token dropping, and the **Switch-Transformer load-balancing loss**.
- Diagnose **routing collapse** with expert-utilization plots, and fix it with the aux loss or **auxiliary-loss-free bias balancing** (DeepSeek-V3 style).
- Explain **fine-grained + shared experts** (DeepSeekMoE) and why they matter.
- Implement a **linear recurrence** both as a sequential loop and as a **parallel associative scan**, and verify they match.
- Build a minimal **selective SSM (Mamba-style, input-dependent Δ, B, C)** and compare it with attention and a non-selective SSM on a **selective-copying** task, including runtime scaling with sequence length.
- Describe **hybrid architectures** (Jamba and 2025-era attention/linear-attention hybrids) accurately.

### 🧰 Prerequisites
Lab 3.1 (Transformer blocks). Comfort with `torch.topk`, `scatter`/`index_add_`, and basic linear algebra. No prior SSM knowledge needed.

### 🌍 Why this matters in 2026
Many of the strongest open-weight models are sparse MoEs (Mixtral, DeepSeek-V3/R1, Qwen3's MoE models, Llama 4, gpt-oss...), so routing and load balancing are now everyday engineering concerns. Meanwhile long-context costs pushed labs toward **hybrids** that interleave a few full-attention layers with many linear-time layers (Mamba-2, gated linear attention / DeltaNet variants). Several production-grade hybrids shipped in 2024–2025. Understanding the scan at the heart of SSMs, and the router at the heart of MoE, lets you reason about these models' speed, memory and failure modes.

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 · Setup | Imports, seeds | 5 |
| A1 · The MoE layer | Top-k routing with expert capacity | 20 |
| A2 · Load balancing | The Switch Transformer auxiliary loss | 15 |
| A3 · Collapse & its cures | Watch routing collapse; fix it with aux loss or DeepSeek-V3 bias balancing | 20 |
| A4 · Fine-grained & shared experts | DeepSeekMoE design ideas | 5 |
| B1 · Linear recurrences | Sequential vs parallel (associative) scan | 20 |
| B2 · Selective SSM | Mamba-style input-dependent Δ, B, C | 20 |
| B3 · Showdown | Attention vs selective SSM vs LTI SSM on selective copying; scaling | 20 |
| B4 · Hybrids | What 2024–2025 hybrid models actually do | 5 |
| Wrap-up | Takeaways, interview corner, boss level | 5 |

## 0 · Setup 🛠️

In [ ]:
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=1337):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})

print(f"torch {torch.__version__} | device = {DEVICE} | FAST = {FAST}")
# No %pip installs needed: everything used here ships with Colab.
# The models in this lab are tiny and launch many small ops, so CPU is used throughout (it's often faster than a GPU here).

# Part A — Mixture-of-Experts 🌼

## A1 · The MoE layer: routing, top-k, capacity

Replace the single MLP in a Transformer block with $E$ expert MLPs $f_1..f_E$ and a **router** $W_r \in \mathbb{R}^{d\times E}$:
$$\pi(x) = \operatorname{softmax}(x W_r),\qquad \mathcal{T}(x) = \operatorname{TopK}(\pi(x), k),\qquad y = \sum_{i \in \mathcal{T}(x)} g_i(x)\, f_i(x).$$
* **Switch Transformer** (Fedus et al., 2022): $k=1$, gate $g_i = \pi_i$.
* **Mixtral / GShard-style**: $k=2$, gates renormalised over the chosen experts, $g_i = \pi_i / \sum_{j\in\mathcal T}\pi_j$.

Parameters grow with $E$, but FLOPs per token grow only with $k$. Mixtral 8×7B has ~47B parameters but uses ~13B per token.

**Capacity.** On hardware, each expert processes a fixed-size buffer. With $N$ tokens in a batch:
$$C = \left\lceil \text{capacity factor} \cdot \frac{N\,k}{E} \right\rceil .$$
Tokens beyond an expert's capacity are **dropped**: they skip the MoE layer (in a real block they still flow through the residual connection). Imbalanced routing ⇒ overflowing experts ⇒ dropped tokens ⇒ worse model. Hence load balancing.

### 🧪 Your Turn #1 — top-k routing with capacity
Implement `topk_route(probs, k, capacity, select_scores=None)`:
1. Choose experts: `topi = (select_scores or probs).topk(k).indices`. (The optional `select_scores` is for Part A3's bias trick: it changes *which* experts are picked, but gate values always come from `probs`.)
2. Gate values `topv = probs.gather(-1, topi)`; if `k > 1` renormalise them to sum to 1.
3. `onehot = F.one_hot(topi, E).float()` → `(N, k, E)`. This is the **pre-capacity assignment**.
4. Capacity: give each token a position in its expert's buffer, processing all first choices before second choices (slot-major order). Flatten to `(k*N, E)` via `onehot.transpose(0, 1).reshape(k * N, E)`, take `cumsum(0) * flat` to get 1-based positions, and keep only entries with position ≤ capacity.
5. Return `gates (N, E)` (combine weight per token-expert, zero if not chosen or dropped) and `assign (N, E)` = pre-capacity assignment summed over k.

In [ ]:
def topk_route(probs, k, capacity, select_scores=None):
    """probs: (N, E) router softmax. Returns gates (N, E) after capacity, assign (N, E) before capacity."""
    N, E = probs.shape
    # TODO 1-2: pick experts and gate values
    # TODO 3: one-hot assignment (N, k, E)
    # TODO 4: capacity via cumulative positions (slot-major)
    # TODO 5: gates and assign
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def topk_route(probs, k, capacity, select_scores=None):
    N, E = probs.shape
    scores = probs if select_scores is None else select_scores
    topi = scores.topk(k, dim=-1).indices                         # (N, k)
    topv = probs.gather(-1, topi)                                 # (N, k)
    if k > 1:
        topv = topv / topv.sum(-1, keepdim=True)
    onehot = F.one_hot(topi, E).float()                           # (N, k, E)
    flat = onehot.transpose(0, 1).reshape(k * N, E)               # slot-major: all 1st choices, then 2nd...
    position = flat.cumsum(0) * flat                              # 1-based slot in expert buffer
    keep = ((position <= capacity).float() * flat).reshape(k, N, E).transpose(0, 1)
    gates = (keep * topv.unsqueeze(-1)).sum(1)                    # (N, E)
    assign = onehot.sum(1)                                        # (N, E)
    return gates, assign

In [ ]:
# ✔️ Checkpoint
probs = torch.tensor([[0.7, 0.2, 0.1],
                      [0.6, 0.3, 0.1],
                      [0.5, 0.1, 0.4],
                      [0.1, 0.1, 0.8]])
g1, a1 = topk_route(probs, k=1, capacity=2)
assert torch.equal(a1, torch.tensor([[1., 0, 0], [1, 0, 0], [1, 0, 0], [0, 0, 1]]))
assert torch.allclose(g1, torch.tensor([[0.7, 0, 0], [0.6, 0, 0], [0, 0, 0], [0, 0, 0.8]])), "token 2 overflows expert 0 and is dropped"
g2, a2 = topk_route(probs, k=2, capacity=10)
assert torch.allclose(g2.sum(-1), torch.ones(4)), "top-2 gates renormalise to 1 when nothing is dropped"
assert torch.allclose(g2[0], torch.tensor([0.7 / 0.9, 0.2 / 0.9, 0.0]))
g3, _ = topk_route(probs, k=1, capacity=2, select_scores=probs + torch.tensor([0.0, 0.0, 0.7]))
assert torch.allclose(g3[0], torch.tensor([0.0, 0.0, 0.1])), "bias changes the choice, but the gate value comes from probs"
print("✔️ routing + capacity correct")

> **🍯 Key takeaway:** Top-k routing gives you E experts' worth of parameters for k experts' worth of FLOPs; capacity limits turn routing imbalance into dropped tokens.

## A2 · The load-balancing loss

Left alone, routers tend to **collapse**: an expert that gets slightly more tokens trains faster, becomes better, and attracts even more tokens. Others starve and die. Switch Transformer adds an auxiliary loss
$$\mathcal{L}_{aux} = \alpha \cdot E \sum_{i=1}^{E} f_i\, P_i,\qquad f_i = \frac{1}{Nk}\sum_{x} \mathbb 1[i \in \mathcal{T}(x)],\qquad P_i = \frac1N\sum_x \pi_i(x).$$
$f_i$ (fraction of tokens dispatched) is not differentiable, but $P_i$ (mean router probability) is, so the gradient pushes probability *away* from overloaded experts. Under perfectly uniform routing, $f_i = P_i = 1/E$ and $E\sum_i f_iP_i = 1$, its minimum value. If everything collapses onto one expert, it equals $E$.

### 🧪 Your Turn #2 — `load_balancing_loss(probs, assign, k)` (without the α)

In [ ]:
def load_balancing_loss(probs, assign, k):
    """probs (N, E), assign (N, E) 0/1 pre-capacity. Returns E * sum_i f_i * P_i."""
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def load_balancing_loss(probs, assign, k):
    N, E = probs.shape
    f = assign.sum(0) / (N * k)
    P = probs.mean(0)
    return E * (f * P).sum()

In [ ]:
# ✔️ Checkpoint
E_ = 8
uniform_p = torch.full((64, E_), 1 / E_)
uniform_a = F.one_hot(torch.arange(64) % E_, E_).float()
assert abs(load_balancing_loss(uniform_p, uniform_a, 1).item() - 1.0) < 1e-6
collapsed_p = torch.zeros(64, E_); collapsed_p[:, 0] = 1
collapsed_a = torch.zeros(64, E_); collapsed_a[:, 0] = 1
assert abs(load_balancing_loss(collapsed_p, collapsed_a, 1).item() - E_) < 1e-6
p = torch.randn(64, E_, requires_grad=True)
pr = p.softmax(-1); _, a = topk_route(pr, 2, 1000)
load_balancing_loss(pr, a, 2).backward()
assert p.grad is not None and p.grad.abs().sum() > 0, "gradient must flow through P"
print("✔️ load-balancing loss: 1 when balanced, E when collapsed, differentiable via P")

Now a full (vectorised-enough) MoE layer. Experts are stored as batched weight tensors `(E, d_in, h)`; for each expert we gather its tokens, run the MLP, and `index_add_` the gated outputs back. (Production kernels sort tokens by expert and run one grouped GEMM instead of a Python loop.)

In [ ]:
class MoE(nn.Module):
    def __init__(self, d_in, d_out, n_experts=8, k=1, capacity_factor=1.25, hidden=32):
        super().__init__()
        self.E, self.k, self.cf = n_experts, k, capacity_factor
        self.router = nn.Linear(d_in, n_experts, bias=False)
        self.w1 = nn.Parameter(torch.randn(n_experts, d_in, hidden) / math.sqrt(d_in))
        self.w2 = nn.Parameter(torch.randn(n_experts, hidden, d_out) / math.sqrt(hidden))
        self.register_buffer("expert_bias", torch.zeros(n_experts))   # used only by bias balancing (A3)

    def forward(self, x, use_bias=False):
        N = x.shape[0]
        probs = self.router(x).softmax(-1)
        capacity = math.ceil(self.cf * N * self.k / self.E)
        gates, assign = topk_route(probs, self.k, capacity,
                                   select_scores=(probs + self.expert_bias) if use_bias else None)
        out = x.new_zeros(N, self.w2.shape[-1])
        for e in range(self.E):
            idx = gates[:, e].nonzero(as_tuple=True)[0]
            if idx.numel() == 0:
                continue
            h = F.gelu(x[idx] @ self.w1[e]) @ self.w2[e]
            out.index_add_(0, idx, h * gates[idx, e:e + 1])
        info = {"aux": load_balancing_loss(probs, assign, self.k),
                "load": (assign.sum(0) / (N * self.k)).detach(),
                "dropped": 1 - (gates > 0).float().sum().item() / (N * self.k)}
        return out, info

### The experiment: does routing collapse?
Task: regress a fixed random "teacher" network, $y = 3\tanh(xW_1)W_2$ with $x\sim\mathcal N(0, I)$. Any expert could learn it, which is exactly the setting where *rich-get-richer* dynamics are free to take over. 8 experts, top-1, capacity factor 1.25.

> **Scale note.** Switch Transformer uses $\alpha = 0.01$ next to a language-modelling loss. Our toy MSE loss has a different scale and far fewer steps, so we use $\alpha = 1$. What matters is the *relative* pull of the two losses.

### 🔮 Predict-then-run
Without any balancing, how many of the 8 experts do you expect to still be receiving tokens after training? What fraction of tokens will be dropped?

In [ ]:
D_IN, D_OUT, N_EXP = 16, 8, 8
gen = torch.Generator().manual_seed(0)
teacher_w1 = torch.randn(D_IN, 64, generator=gen) / 4
teacher_w2 = torch.randn(64, D_OUT, generator=gen) / 8
def moe_data(n):
    x = torch.randn(n, D_IN)
    return x, 3 * torch.tanh(x @ teacher_w1) @ teacher_w2

MOE_STEPS = 100 if FAST else 1200

def train_moe(mode, alpha=1.0, gamma=1e-3, steps=MOE_STEPS, k=1, cf=1.25):
    seed_everything(1)
    m = MoE(D_IN, D_OUT, N_EXP, k=k, capacity_factor=cf)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    hist = {"load": [], "dropped": [], "mse": []}
    for step in range(steps):
        x, y = moe_data(512)
        out, info = m(x, use_bias=(mode == "bias"))
        mse = F.mse_loss(out, y)
        loss = mse + (alpha * info["aux"] if mode == "aux" else 0.0)
        opt.zero_grad(); loss.backward(); opt.step()
        if mode == "bias":
            update_expert_bias(m.expert_bias, info["load"], gamma)
        hist["load"].append(info["load"].numpy()); hist["dropped"].append(info["dropped"]); hist["mse"].append(mse.item())
    with torch.no_grad():
        x, y = moe_data(4096); out, info = m(x, use_bias=(mode == "bias"))
    hist["test_mse"], hist["test_dropped"] = F.mse_loss(out, y).item(), info["dropped"]
    return m, {k_: np.array(v) if isinstance(v, list) else v for k_, v in hist.items()}

> **🍯 Key takeaway:** The Switch aux loss E·Σ f_i P_i equals 1 under uniform routing and E under total collapse; its gradient flows through the router probabilities P.

## A3 · Auxiliary-loss-free balancing (DeepSeek-V3)

The aux loss works, but it adds a gradient that *competes* with the real objective. DeepSeek-V3 (2024) instead adds a per-expert **bias** $b_i$ to the routing scores *only when choosing* the top-k experts (the gate values still use the unbiased scores), and nudges it after every step:
$$b_i \leftarrow b_i + \gamma\cdot\operatorname{sign}(\bar f - f_i).$$
Overloaded experts become less attractive, underloaded ones more, with no gradient interference. (DeepSeek-V3 uses sigmoid affinities and keeps a very small sequence-level balance loss as a safety net; we keep the idea and simplify the rest.)

### 🧪 Your Turn #3 — the bias update (in place)

In [ ]:
@torch.no_grad()
def update_expert_bias(bias, load, gamma):
    """bias, load: (E,). Increase bias of under-loaded experts, decrease over-loaded ones, in place."""
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
@torch.no_grad()
def update_expert_bias(bias, load, gamma):
    bias += gamma * torch.sign(load.mean() - load)

In [ ]:
# ✔️ Checkpoint
b = torch.zeros(4)
update_expert_bias(b, torch.tensor([0.4, 0.3, 0.2, 0.1]), 0.01)
assert torch.allclose(b, torch.tensor([-0.01, -0.01, 0.01, 0.01]))
print("✔️ bias update correct")

runs = {}
for mode in ["none", "aux", "bias"]:
    t0 = time.time()
    _, runs[mode] = train_moe(mode)
    r = runs[mode]
    print(f"{mode:5s} | test MSE {r['test_mse']:.3f} | dropped {100 * r['test_dropped']:4.1f}% | "
          f"dead experts {(r['load'][-50:].mean(0) < 0.01).sum()} | {time.time() - t0:.0f}s")

In [ ]:
titles = {"none": "no balancing", "aux": "Switch aux loss (α=1)", "bias": "aux-loss-free bias (γ=1e-3)"}
fig, axes = plt.subplots(2, 3, figsize=(16, 7), gridspec_kw={"height_ratios": [2, 1]})
for c, mode in enumerate(["none", "aux", "bias"]):
    r = runs[mode]
    im = axes[0, c].imshow(r["load"].T, aspect="auto", cmap="YlOrBr", vmin=0, vmax=0.4, interpolation="nearest")
    axes[0, c].set_title(f"{titles[mode]}\ntest MSE {r['test_mse']:.3f}, dropped {100 * r['test_dropped']:.0f}%")
    axes[0, c].set_xlabel("step"); axes[0, c].set_ylabel("expert")
    axes[1, c].bar(range(N_EXP), r["load"][-50:].mean(0), color=HIVE[0], edgecolor=HIVE[1])
    axes[1, c].axhline(1 / N_EXP, ls="--", color=HIVE[2], label="uniform 1/E"); axes[1, c].set_ylim(0, 0.45)
    axes[1, c].set_xlabel("expert"); axes[1, c].set_ylabel("final load")
axes[1, 0].legend()
plt.colorbar(im, ax=axes[0, :].tolist(), fraction=0.02, label="fraction of tokens routed")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
for mode in runs:
    ax.plot(np.convolve(runs[mode]["dropped"], np.ones(20) / 20, mode="valid"), label=titles[mode])
ax.set_ylabel("dropped-token fraction"); ax.set_xlabel("step"); ax.legend(); ax.set_title("imbalance → overflowing experts → dropped tokens")
plt.show()

### 🎬 The routing race
Same three runs, replayed. Each bar is one expert's share of tokens (smoothed over 25 steps). The dashed line is perfect balance and the red line is the **capacity limit**: anything above it gets dropped. Watch the unbalanced router squeeze experts out one by one.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

cap_share = math.ceil(1.25 * 512 / N_EXP) / 512          # capacity as a fraction of the batch (top-1)
smooth = {m: np.array([runs[m]["load"][max(0, i - 25):i + 1].mean(0) for i in range(len(runs[m]["load"]))]) for m in runs}
frames = np.linspace(0, MOE_STEPS - 1, 40 if not FAST else 10).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
def draw(fi):
    t = frames[fi]
    for ax, mode in zip(axes, ["none", "aux", "bias"]):
        ax.clear()
        load = smooth[mode][t]
        ax.bar(range(N_EXP), load, color=[HIVE[2] if l > cap_share else HIVE[0] for l in load], edgecolor=HIVE[1])
        ax.axhline(1 / N_EXP, ls="--", color=HIVE[1], lw=1)
        ax.axhline(cap_share, color=HIVE[2], lw=1.5)
        ax.set_ylim(0, 0.42); ax.set_xticks(range(N_EXP)); ax.set_xlabel("expert")
        ax.set_title(f"{titles[mode]}\nstep {t}  ·  dead experts: {(load < 0.01).sum()}", fontsize=10)
    axes[0].set_ylabel("share of tokens")
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=200)
plt.close(fig)
HTML(anim.to_jshtml())

**What to look for.** Without balancing, a few experts die early (their rows fade to pale) and the survivors overflow their capacity, so a big chunk of tokens is dropped and the test error is worse. Both balancing methods keep all experts alive and nearly eliminate drops. On this toy problem the two fixes are similar; at LLM scale DeepSeek reported that the bias method gave better model quality than a strong aux loss because it doesn't fight the main objective.

> **🍯 Key takeaway:** Routers collapse (rich get richer) unless you push back. An aux loss or a simple per-expert bias nudge keeps every expert alive and stops token dropping.

---
## A4 · Fine-grained & shared experts (DeepSeekMoE → DeepSeek-V3)

Two design ideas from **DeepSeekMoE** (Dai et al., 2024), used at scale in DeepSeek-V2 and V3:
1. **Fine-grained expert segmentation.** Split each expert into $m$ smaller ones (hidden size ÷ $m$) and activate $m\cdot k$ of them. FLOPs and parameters are unchanged, but the number of possible expert *combinations* explodes, so experts can specialise more narrowly.
2. **Shared experts.** A few experts are *always* active for every token, absorbing common knowledge (syntax, frequent tokens) so routed experts don't all waste capacity re-learning it.

**DeepSeek-V3** (Dec 2024): 671B total parameters, ~37B activated per token; each MoE layer has **1 shared + 256 routed experts, 8 routed active per token**, with the auxiliary-loss-free bias balancing you just implemented.

In [ ]:
for name, (E_tot, k_act) in {"16 experts, top-2 (coarse)": (16, 2), "64 experts, top-8 (each 1/4 the size)": (64, 8),
                             "256 experts, top-8 (DeepSeek-V3 routed)": (256, 8)}.items():
    print(f"{name:42s}: {math.comb(E_tot, k_act):.3e} possible expert combinations per token")

### 🧪 (Bonus-sized) Your Turn — add a shared expert
Modify `MoE` so it has one extra always-on expert (a plain MLP applied to every token, added to the routed output). Re-run `train_moe("aux")` and compare test MSE. *(No solution cell: this one's yours. Hint: add `self.shared = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Linear(hidden, d_out))` and `out = out + self.shared(x)`.)*

> **🍯 Key takeaway:** Many small experts plus a few always-on shared experts give more specialisation for the same compute; that's the DeepSeekMoE/V3 recipe.

# Part B — Linear recurrences & selective state-space models 🐍

## B1 · Linear recurrences and the parallel scan

Attention at step $t$ looks at all $t$ previous tokens: $O(t)$ work and memory per token, $O(T^2)$ total. A **recurrence** compresses the past into a fixed-size state:
$$h_t = a_t \odot h_{t-1} + b_t .$$
Generation costs $O(1)$ per token. But training with a for-loop over $T$ is sequential and slow on parallel hardware... *unless* the recurrence is **linear** in $h$. Then it is a **scan** with an *associative* operator. Represent step $t$ as the pair $(a_t, b_t)$ meaning "$h \mapsto a_t h + b_t$". Composing two steps:
$$(a_1, b_1) \bullet (a_2, b_2) = (a_1 a_2,\; a_2 b_1 + b_2),$$
since $a_2(a_1 h + b_1) + b_2 = (a_1a_2)h + (a_2b_1 + b_2)$. Associativity means we can combine in any bracketing, e.g. a tree, giving $O(\log T)$ *parallel* steps (Blelloch, 1990; Martin & Cundy, 2018).

**Hillis–Steele scan:** for $s = 1, 2, 4, \dots$: every position $t \ge s$ combines with position $t-s$. After $\lceil\log_2 T\rceil$ rounds, position $t$ holds the composition of steps $0..t$, i.e. $h_t$ (with $h_{-1}=0$).

### 🧪 Your Turn #4 — sequential scan (the ground truth)
### 🧪 Your Turn #5 — parallel scan
For `parallel_scan`, at each round build shifted copies: `a_prev` = `a` shifted right by `s` along time, padded with the identity element **a = 1**; `b_prev` padded with **b = 0**. Then update `b = a * b_prev + b` and `a = a * a_prev` (order matters: use the *old* `a` when updating `b`).
*Hint:* `torch.cat([torch.ones_like(a[:, :s]), a[:, :-s]], dim=1)`.

In [ ]:
def sequential_scan(a, b):
    """a, b: (B, T, ...). Returns h with h[:, t] = a[:, t] * h[:, t-1] + b[:, t], h[:, -1] = 0."""
    # TODO: loop over t
    raise NotImplementedError

def parallel_scan(a, b):
    """Same result in ceil(log2 T) vectorised rounds (Hillis-Steele)."""
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def sequential_scan(a, b):
    h = torch.zeros_like(b[:, 0])
    out = []
    for t in range(a.shape[1]):
        h = a[:, t] * h + b[:, t]
        out.append(h)
    return torch.stack(out, dim=1)

def parallel_scan(a, b):
    T = a.shape[1]
    s = 1
    while s < T:
        a_prev = torch.cat([torch.ones_like(a[:, :s]), a[:, :-s]], dim=1)
        b_prev = torch.cat([torch.zeros_like(b[:, :s]), b[:, :-s]], dim=1)
        b = a * b_prev + b          # uses the old a
        a = a * a_prev
        s *= 2
    return b

In [ ]:
# ✔️ Checkpoint
torch.manual_seed(0)
a = torch.rand(3, 37, 5) * 0.99
b = torch.randn(3, 37, 5)
h_seq, h_par = sequential_scan(a, b), parallel_scan(a, b)
assert torch.allclose(h_seq, h_par, atol=1e-5)
# closed form for constant a: h_t = sum_s a^(t-s) b_s
a_c = torch.full((1, 6, 1), 0.5); b_c = torch.ones(1, 6, 1)
expected = torch.tensor([1, 1.5, 1.75, 1.875, 1.9375, 1.96875])
assert torch.allclose(sequential_scan(a_c, b_c).flatten(), expected), "sequential scan disagrees with the closed form"
assert torch.allclose(parallel_scan(a_c, b_c).flatten(), expected), "parallel scan disagrees with the closed form"
# gradients flow through the parallel version too
a_g = a.clone().requires_grad_(); parallel_scan(a_g, b).sum().backward(); assert a_g.grad is not None
print("✔️ parallel scan == sequential scan")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
T_ = 60
for a_val in [0.5, 0.9, 0.98]:
    impulse = torch.zeros(1, T_, 1); impulse[0, 5] = 1
    axes[0].plot(parallel_scan(torch.full((1, T_, 1), a_val), impulse).flatten(), label=f"a = {a_val}")
axes[0].set_xlabel("time"); axes[0].set_title("impulse at t=5: a controls how long the state remembers"); axes[0].legend()

Ts = [64, 256, 1024, 4096] if not FAST else [64, 256]
ts_seq, ts_par = [], []
for T_ in Ts:
    a = torch.rand(8, T_, 64) * 0.99; b = torch.randn(8, T_, 64)
    t0 = time.perf_counter(); sequential_scan(a, b); ts_seq.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); parallel_scan(a, b); ts_par.append(time.perf_counter() - t0)
axes[1].loglog(Ts, ts_seq, "o-", label="sequential (T Python steps)")
axes[1].loglog(Ts, ts_par, "s-", label="parallel (log₂T vectorised rounds)")
axes[1].set_xlabel("sequence length T"); axes[1].set_ylabel("seconds"); axes[1].legend(); axes[1].set_title("scan runtime")
plt.tight_layout(); plt.show()

Hillis–Steele does $O(T\log T)$ total work (Blelloch's up/down-sweep does $O(T)$), but its $\log T$ depth is what matters on parallel hardware. Mamba's CUDA kernel goes further: it fuses the scan with discretisation and keeps the expanded state in on-chip SRAM, never writing the big $(B, T, D, N)$ tensor to HBM, the same IO-awareness idea as FlashAttention. Mamba-2 recasts the computation as chunked matrix multiplications ("state-space duality") to use tensor cores.

> **🍯 Key takeaway:** A linear recurrence is an associative scan: O(log T) parallel depth for training, O(1) state for generation.

---
## B2 · A selective SSM (Mamba-style)

A (diagonal) **state-space model** maps an input channel $u(t)$ through an $N$-dimensional state:
$$h'(t) = A\,h(t) + B\,u(t),\qquad y(t) = C\,h(t) + D\,u(t).$$
Discretising with step size $\Delta$ (zero-order hold for $A$, a simple Euler step for $B$, as in Mamba's reference code):
$$\bar A = \exp(\Delta A),\quad \bar B = \Delta B,\qquad h_t = \bar A\, h_{t-1} + \bar B\, u_t,\quad y_t = C h_t + D u_t .$$
That's exactly our linear recurrence with $a_t = \bar A$ and $b_t = \bar B u_t$.

* **S4 / LTI SSMs:** $\Delta, B, C$ are fixed parameters, the same at every time step (*linear time-invariant*). Then the whole layer is a long convolution, but it can't decide *per token* what to remember.
* **Mamba's selection mechanism (S6):** make $\Delta_t, B_t, C_t$ **functions of the input** $x_t$. A large $\Delta_t$ means "reset and write this token into the state"; a tiny $\Delta_t$ means "ignore this token and keep the state". It is a learned, input-dependent gate, reminiscent of LSTM gates, but still a linear recurrence in $h$, so the parallel scan still applies.

Shapes for one layer with $D$ channels and state size $N$: $\Delta_t \in \mathbb R^{D}$, $A \in \mathbb R^{D\times N}$ (negative, diagonal), $B_t, C_t \in \mathbb R^{N}$ (shared across channels), state $h_t \in \mathbb R^{D\times N}$.

### 🧪 Your Turn #6 — the selective scan, sequential reference
Given `u (Bt, T, D)`, `delta (Bt, T, D)`, `A (D, N)`, `Bm (Bt, T, N)`, `Cm (Bt, T, N)`, `Dskip (D,)`:
loop over $t$: `dA = exp(delta[:, t, :, None] * A)`, `dBu = delta[:, t, :, None] * Bm[:, t, None, :] * u[:, t, :, None]`, `h = dA * h + dBu`, `y_t = (h * Cm[:, t, None, :]).sum(-1) + Dskip * u[:, t]`.

In [ ]:
def selective_scan_ref(u, delta, A, Bm, Cm, Dskip):
    Bt, T, D = u.shape
    N = A.shape[1]
    h = u.new_zeros(Bt, D, N)
    ys = []
    # TODO: the loop described above; return torch.stack(ys, dim=1) of shape (Bt, T, D)
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def selective_scan_ref(u, delta, A, Bm, Cm, Dskip):
    Bt, T, D = u.shape
    N = A.shape[1]
    h = u.new_zeros(Bt, D, N)
    ys = []
    for t in range(T):
        dA = torch.exp(delta[:, t, :, None] * A)                              # (Bt, D, N)
        dBu = delta[:, t, :, None] * Bm[:, t, None, :] * u[:, t, :, None]      # (Bt, D, N)
        h = dA * h + dBu
        ys.append((h * Cm[:, t, None, :]).sum(-1) + Dskip * u[:, t])
    return torch.stack(ys, dim=1)

In [ ]:
def selective_scan_parallel(u, delta, A, Bm, Cm, Dskip):
    """Same computation, all time steps at once via the associative scan."""
    dA = torch.exp(delta.unsqueeze(-1) * A)                                    # (Bt, T, D, N)
    dBu = delta.unsqueeze(-1) * Bm.unsqueeze(2) * u.unsqueeze(-1)              # (Bt, T, D, N)
    h = parallel_scan(dA, dBu)
    return (h * Cm.unsqueeze(2)).sum(-1) + Dskip * u

# ✔️ Checkpoint
torch.manual_seed(0)
Bt, T, D, N = 2, 33, 6, 4
u = torch.randn(Bt, T, D); delta = F.softplus(torch.randn(Bt, T, D))
A = -torch.rand(D, N) * 2; Bm = torch.randn(Bt, T, N); Cm = torch.randn(Bt, T, N); Dskip = torch.randn(D)
y_ref = selective_scan_ref(u, delta, A, Bm, Cm, Dskip)
y_par = selective_scan_parallel(u, delta, A, Bm, Cm, Dskip)
assert y_ref.shape == (Bt, T, D) and torch.allclose(y_ref, y_par, atol=1e-4)
print("✔️ selective scan: sequential == parallel")

### The Mamba block
Around the scan, Mamba wraps a gated block (simplified from Gu & Dao, 2023):
```
x ─► Linear (d → 2·d_inner) ─► split ─► u ─► causal depthwise conv (k=4) ─► SiLU ─► selective SSM ─┐
                                    └──► z ─────────────────────────────────────────► SiLU ───────── ⊙ ─► Linear (d_inner → d)
```
$\Delta_t$ is produced through a low-rank projection followed by softplus (keeping it positive), and $A$ is parameterised as $-\exp(A_{log})$ (keeping it negative, so the recurrence is stable: $|\bar A| < 1$). We implement the depthwise conv as a sum of shifted copies: identical maths to `nn.Conv1d(groups=d_inner)` but much faster for tiny kernels on CPU. A `selective=False` switch turns the block into an LTI SSM (fixed $\Delta, B, C$) for comparison.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def init_dt_bias(n, dt_min=1e-3, dt_max=1e-1):
    dt = torch.exp(torch.rand(n) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
    return dt + torch.log(-torch.expm1(-dt))          # inverse softplus, so softplus(bias) = dt

class MambaBlock(nn.Module):
    def __init__(self, d, d_state=16, expand=2, conv_k=4, selective=True):
        super().__init__()
        di = expand * d
        self.di, self.N, self.K, self.selective = di, d_state, conv_k, selective
        self.in_proj = nn.Linear(d, 2 * di, bias=False)
        self.conv_w = nn.Parameter(torch.randn(di, conv_k) / math.sqrt(conv_k)); self.conv_b = nn.Parameter(torch.zeros(di))
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float().repeat(di, 1)))   # S4D-real init
        self.Dskip = nn.Parameter(torch.ones(di))
        if selective:
            self.dt_rank = max(1, d // 16)
            self.x_proj = nn.Linear(di, self.dt_rank + 2 * d_state, bias=False)   # -> (dt_low_rank, B, C)
            self.dt_proj = nn.Linear(self.dt_rank, di)
            with torch.no_grad(): self.dt_proj.bias.copy_(init_dt_bias(di))
        else:                                   # LTI: one Delta per channel, one B and C for all time steps
            self.dt_raw = nn.Parameter(init_dt_bias(di))
            self.B_lti = nn.Parameter(torch.randn(d_state) / math.sqrt(d_state))
            self.C_lti = nn.Parameter(torch.randn(d_state) / math.sqrt(d_state))
        self.out_proj = nn.Linear(di, d, bias=False)
        self.record = False; self.last_delta = None

    def forward(self, x):
        Bt, T, _ = x.shape
        u, z = self.in_proj(x).chunk(2, dim=-1)
        up = F.pad(u, (0, 0, self.K - 1, 0))                                         # causal padding in time
        u = F.silu(sum(up[:, j:j + T] * self.conv_w[:, j] for j in range(self.K)) + self.conv_b)
        A = -torch.exp(self.A_log)
        if self.selective:
            dt, Bm, Cm = self.x_proj(u).split([self.dt_rank, self.N, self.N], dim=-1)
            delta = F.softplus(self.dt_proj(dt))                                      # (Bt, T, di), input-dependent!
        else:
            delta = F.softplus(self.dt_raw).expand(Bt, T, self.di)
            Bm, Cm = self.B_lti.expand(Bt, T, self.N), self.C_lti.expand(Bt, T, self.N)
        if self.record: self.last_delta = delta.detach()
        y = selective_scan_parallel(u, delta, A, Bm, Cm, self.Dskip)
        return self.out_proj(y * F.silu(z))

For the attention baseline we reuse a compact Lab 3.1-style block (RoPE + causal SDPA, followed by a plain GELU MLP). Both model families get 2 layers and d = 64. Note that the Mamba models have **no MLP and no positional encoding**: the recurrence itself knows about order.

In [ ]:
def rope_angles(T, dh, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dh, 2).float() / dh))
    ang = torch.outer(torch.arange(T).float(), inv)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

class AttnBlock(nn.Module):
    def __init__(self, d, n_head=4):
        super().__init__()
        self.H = n_head; self.qkv = nn.Linear(d, 3 * d, bias=False); self.proj = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.H, C // self.H).permute(2, 0, 3, 1, 4)
        cos, sin = rope_angles(T, C // self.H)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, C)
        return self.proj(y)

class SeqModel(nn.Module):
    def __init__(self, kind, vocab, d=64, n_layer=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        make = {"attention": lambda: AttnBlock(d), "selective SSM": lambda: MambaBlock(d, selective=True),
                "LTI SSM": lambda: MambaBlock(d, selective=False)}[kind]
        self.layers = nn.ModuleList([make() for _ in range(n_layer)])
        self.norms = nn.ModuleList([RMSNorm(d) for _ in range(n_layer)])
        # Transformers interleave attention with MLPs; Mamba blocks are used on their own (no separate MLP).
        self.mlps = nn.ModuleList([nn.Sequential(RMSNorm(d), nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
                                   for _ in range(n_layer)]) if kind == "attention" else None
        self.norm_f = RMSNorm(d); self.head = nn.Linear(d, vocab)
    def forward(self, x):
        h = self.emb(x)
        for i, (norm, layer) in enumerate(zip(self.norms, self.layers)):
            h = h + layer(norm(h))
            if self.mlps is not None:
                h = h + self.mlps[i](h)
        return self.head(self.norm_f(h))

> **🍯 Key takeaway:** Mamba = linear recurrence + input-dependent Δ, B, C. The model decides per token what to write into its state and what to forget.

## B3 · Showdown on selective copying 🥊

**Selective copying** (from the Mamba paper): a sequence of length $L$ is mostly blank (`·`) with $M=4$ data tokens at **random** positions; after the context come $M$ query markers (`?`), and the model must output the data tokens in order.
```
input :  · · 3 · · · · 7 · · 1 · · · · · · 5 · ·  ? ? ? ?
target:                                            3 7 1 5
```
Because the positions are random, a fixed convolution (an LTI SSM is a long convolution) can't know where to look. You need **content-dependent** behaviour: attention can look things up; a selective SSM can choose to write data tokens into its state and ignore blanks.

### 🔮 Predict-then-run
Rank the three models (attention, selective SSM, LTI SSM) by final accuracy. Which one do you expect to be fastest to train *on this CPU*?

In [ ]:
N_DATA_TOK, M_COPY = 8, 4
BLANK, QUERY = 0, N_DATA_TOK + 1
VOCAB_SC = N_DATA_TOK + 2

def selective_copy_batch(B, L, gen=None):
    x = torch.full((B, L + M_COPY), BLANK, dtype=torch.long)
    data = torch.randint(1, N_DATA_TOK + 1, (B, M_COPY), generator=gen)
    pos = torch.rand(B, L, generator=gen).argsort(-1)[:, :M_COPY].sort(-1).values
    x.scatter_(1, pos, data)
    x[:, L:] = QUERY
    return x, data

xs, ys = selective_copy_batch(2, 20)
show = lambda seq: " ".join({BLANK: "·", QUERY: "?"}.get(int(t), str(int(t))) for t in seq)
for i in range(2):
    print("input :", show(xs[i]), "\ntarget:", ys[i].tolist())

In [ ]:
L_TRAIN = 32
SC_STEPS = 20 if FAST else 600

def train_seq_model(kind, steps=SC_STEPS, L=L_TRAIN, B=64, lr=2e-3):
    seed_everything(0)
    m = SeqModel(kind, VOCAB_SC)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.0)
    losses = []
    t0 = time.time()
    for step in range(steps):
        x, y = selective_copy_batch(B, L)
        logits = m(x)[:, -M_COPY:]
        loss = F.cross_entropy(logits.reshape(-1, VOCAB_SC), y.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        losses.append(loss.item())
    return m, np.array(losses), time.time() - t0

@torch.no_grad()
def copy_accuracy(m, L, n=512):
    x, y = selective_copy_batch(n, L, gen=torch.Generator().manual_seed(123))
    return (m(x)[:, -M_COPY:].argmax(-1) == y).float().mean().item()

sc_models, sc_curves = {}, {}
for kind in ["attention", "selective SSM", "LTI SSM"]:
    m, curve, secs = train_seq_model(kind)
    sc_models[kind], sc_curves[kind] = m, curve
    n_par = sum(p.numel() for p in m.parameters())
    print(f"{kind:14s} | params {n_par:7,} | train {secs:5.1f}s | accuracy @L={L_TRAIN}: {copy_accuracy(m, L_TRAIN):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for kind, c in sc_curves.items():
    axes[0].plot(np.convolve(c, np.ones(10) / 10, mode="valid"), label=kind)
axes[0].axhline(math.log(N_DATA_TOK), ls=":", color="gray", label="chance (ln 8)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("selective copying: training loss")
L_test = [32, 64, 128] if FAST else [32, 64, 128, 256]
for kind, m in sc_models.items():
    axes[1].plot(L_test, [copy_accuracy(m, L) for L in L_test], "o-", label=kind)
axes[1].axhline(1 / N_DATA_TOK, ls=":", color="gray", label="chance")
axes[1].set_xscale("log", base=2); axes[1].set_xlabel("context length L at test time (trained at 32)")
axes[1].set_ylabel("token accuracy"); axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].set_title("length generalisation")
plt.tight_layout(); plt.show()

**Interpretation guide.** On the training length, attention and the selective SSM should both get close to perfect, while the LTI SSM plateaus well below (its gating and short convolution give it *some* content-dependence, but its recurrence can't choose what to store). The right-hand panel tests **longer** sequences than seen in training. In our reference run the selective SSM stayed near-perfect at 2× and 4× the training length, while RoPE attention dropped sharply at 2×: it has never seen the larger relative offsets, whereas the selective SSM only needs its "ignore blanks, store data" rule to keep working. This mirrors the Mamba paper's report of strong length extrapolation for selective SSMs on synthetic tasks such as induction heads. Results on a tiny run vary by seed, so treat one run as an anecdote, not a benchmark (and attention's extrapolation can be improved with other position schemes).

### Peeking at selection: what does Δ do?

In [ ]:
m_sel = sc_models["selective SSM"]
x, y = selective_copy_batch(1, 48, gen=torch.Generator().manual_seed(7))
for blk in m_sel.layers: blk.record = True
with torch.no_grad(): m_sel(x)
for blk in m_sel.layers: blk.record = False
data_pos = [i for i, t in enumerate(x[0].tolist()) if 1 <= t <= N_DATA_TOK]
fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
for li, ax in enumerate(axes):
    d = m_sel.layers[li].last_delta[0].T                                  # (d_inner, T)
    rel = torch.log(d / d.median(dim=1, keepdim=True).values)             # per-channel deviation from typical Δ
    order = rel.abs().mean(1).argsort(descending=True)[:64]               # the 64 most input-sensitive channels
    im = ax.imshow(rel[order].numpy(), aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5, interpolation="nearest")
    for p_ in data_pos: ax.axvline(p_, color=HIVE[1], lw=0.6, ls=":")
    ax.set_ylabel(f"layer {li}\nchannels"); ax.set_yticks([])
axes[1].set_xticks(range(x.shape[1])); axes[1].set_xticklabels([show([t]) for t in x[0]], fontsize=8)
plt.colorbar(im, ax=axes.tolist(), fraction=0.02, label="log(Δ / channel median)")
axes[0].set_title("Δ per channel and position (dotted lines = data tokens). An LTI SSM would show identical columns everywhere.")
plt.show()

In **layer 0**, every blank looks identical once its 4-wide convolution window contains only blanks, so those columns share exactly the same Δ pattern (flat white). Around each data token (and at the queries) the columns change sharply: the step size, and hence how much the state forgets vs. writes, is **chosen per token from the content**. Which channels go up and which go down is the model's own learned bookkeeping; the point is that it *can* treat tokens differently, which an LTI SSM cannot. In **layer 1** the inputs already carry a summary of the past, so Δ also shifts with *what has been stored so far* (note the change after the second data token).

### Runtime & memory scaling with sequence length
Asymptotically, attention costs $O(T^2)$ per layer in training and its KV cache grows as $O(T)$ during generation; an SSM costs $O(T)$ (here $O(T\log T)$ with Hillis–Steele) and keeps an $O(1)$ state. Constants matter a lot in practice, so let's measure our implementations honestly.

In [ ]:
d_bench = 64
attn_layer, ssm_layer = AttnBlock(d_bench).eval(), MambaBlock(d_bench).eval()
Ls = [128, 256, 512] if FAST else [256, 512, 1024, 2048, 4096]

def bench(fn, reps=3):
    fn(); t0 = time.perf_counter()
    for _ in range(reps): fn()
    return (time.perf_counter() - t0) / reps

def naive_attention_forward(x, layer=attn_layer):      # materialises the T x T matrix (what FlashAttention avoids)
    B, T, C = x.shape
    q, k, v = layer.qkv(x).view(B, T, 3, layer.H, C // layer.H).permute(2, 0, 3, 1, 4)
    s = (q @ k.transpose(-2, -1)) / math.sqrt(C // layer.H)
    s = s.masked_fill(torch.ones(T, T, dtype=torch.bool).triu(1), float("-inf"))
    return s.softmax(-1) @ v

times = {"attention (SDPA)": [], "attention (naive T×T)": [], "selective SSM (parallel scan)": []}
with torch.no_grad():
    for L in Ls:
        x = torch.randn(1, L, d_bench)
        times["attention (SDPA)"].append(bench(lambda: attn_layer(x)))
        times["attention (naive T×T)"].append(bench(lambda: naive_attention_forward(x)))
        times["selective SSM (parallel scan)"].append(bench(lambda: ssm_layer(x)))
        print(f"L={L:5d} | " + " | ".join(f"{k}: {v[-1] * 1e3:7.1f} ms" for k, v in times.items()))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for k, v in times.items():
    axes[0].loglog(Ls, v, "o-", label=k)
ref = np.array(Ls, dtype=float)
axes[0].loglog(Ls, times["attention (SDPA)"][0] * (ref / Ls[0]) ** 2, ":", color="gray", label="∝ T² reference")
axes[0].loglog(Ls, times["selective SSM (parallel scan)"][0] * (ref / Ls[0]), "--", color="gray", label="∝ T reference")
axes[0].set_xlabel("sequence length"); axes[0].set_ylabel("forward time (s), batch 1, CPU"); axes[0].legend(fontsize=8)
axes[0].set_title("one layer, forward pass")

ctx = np.array([1e3, 4e3, 16e3, 64e3, 256e3, 1e6])
kv_bytes = 2 * 32 * 8 * 128 * 2 * ctx              # Llama-3-8B-like KV cache (32 layers, 8 KV heads, d_head 128, fp16)
ssm_state = np.full_like(ctx, 64 * (2 * 4096) * 128 * 2)  # Mamba-2-like: 64 layers, d_inner 8192, N=128, fp16 (illustrative)
axes[1].loglog(ctx, kv_bytes / 2**30, "o-", label="attention KV cache (8B-class GQA model)")
axes[1].loglog(ctx, ssm_state / 2**30, "s-", label="SSM recurrent state (constant)")
axes[1].set_xlabel("context length (tokens)"); axes[1].set_ylabel("GiB per sequence"); axes[1].legend(fontsize=8)
axes[1].set_title("generation-time memory (analytical, illustrative configs)")
plt.tight_layout(); plt.show()

**Reading it honestly.** PyTorch's fused SDPA is extremely well optimised, while our scan is a pure-Python-orchestrated $O(T\log T)$ algorithm that materialises a $(T, d_{inner}, N)$ tensor, so at short lengths attention may well be *faster*. The **slopes** are the story: naive attention grows quadratically, and the SSM grows (near-)linearly. At generation time the difference is starker: attention must keep and re-read a KV cache that grows with every token, while an SSM carries a fixed-size state. That is the whole motivation for hybrids.

> **🍯 Key takeaway:** Selection matters. Content-dependent tasks defeat LTI SSMs; the selective SSM matches attention here and extrapolates to longer inputs, while attention's edge is exact recall.

---
## B4 · Hybrids: the 2024–2025 consensus 🧬

Pure SSMs/linear-attention models compress the past into a fixed state, so they are weaker at **exact recall** of arbitrary earlier tokens (e.g. copying a phone number from 50k tokens ago, or associative recall, as analysed in the *Zoology* paper, Arora et al., 2023). Attention is exact but expensive. The pragmatic answer: **mix them**, with a few attention layers for precise retrieval and many linear-time layers for cheap long-range mixing.

| Model (year) | Recipe (as reported by the authors) |
|---|---|
| **Jamba** (AI21, 2024) | Interleaves Transformer and Mamba layers (1 attention : 7 Mamba), with MoE MLPs in every other layer; 52B total / 12B active; 256K context. |
| **Samba** (Microsoft, 2024) | Mamba layers interleaved with *sliding-window* attention. |
| **Zamba** (Zyphra, 2024) | Mamba backbone with a *shared* attention block reused at several depths. |
| **MiniMax-01** (2025) | "Lightning attention" (a linear-attention variant) with one softmax-attention layer after every 7 linear layers, plus MoE; very long context. |
| **Nemotron-H** (NVIDIA, 2025) | Replaces most self-attention layers in a Transformer with Mamba-2 layers. |
| **Falcon-H1** (TII, 2025) | *Parallel* hybrid: attention and Mamba-2 heads side by side within each layer. |
| **Qwen3-Next** (Alibaba, 2025) | Gated DeltaNet (linear attention with a delta-rule update) and gated full attention in a 3:1 ratio, with a very sparse MoE. |

The common thread: **mostly linear-time layers, a minority of full-attention layers**, often combined with MoE. Mamba-2's "state-space duality" result (Dao & Gu, 2024) also showed that selective SSMs and gated linear attention are two views of the same family, which is why "SSM" and "linear attention" increasingly blur together.

> **🍯 Key takeaway:** Mostly linear-time layers plus a few full-attention layers (often with MoE MLPs) is the 2024–2025 recipe for efficient long context.

## 🏆 Boss Level
1. **Associative recall (MQAR).** Build the *multi-query associative recall* task from *Zoology* (key–value pairs, then several queried keys) and compare attention vs selective SSM as you shrink the SSM's state size `d_state ∈ {4, 8, 16, 32}`. Plot accuracy vs state size and explain it with the "fixed-size memory" argument.
2. **A hybrid.** Build a 3-layer model `[Mamba, Attention, Mamba]` and compare it to 3×Mamba and 3×Attention on selective copying *and* associative recall at matched parameter count. Does one attention layer close the recall gap?
3. **MoE inside a Transformer.** Replace the MLP in Lab 3.1's GPT with your `MoE` layer (top-2, 8 experts, aux loss α=0.01) and train on Tiny Shakespeare. Compare val loss vs a dense model with the same *active* parameters, and plot per-layer expert utilisation. Bonus: which characters does each expert specialise in?

## 🍯 Key Takeaways
- **MoE** decouples parameter count from per-token compute: each token visits only $k$ of $E$ experts.
- **Capacity** bounds each expert's buffer, so imbalance means **dropped tokens**; routing collapse is a real failure you can *see* in utilisation plots.
- Fix it with the **Switch aux loss** $E\sum_i f_iP_i$ or DeepSeek-V3's **aux-loss-free bias** nudges; **fine-grained + shared experts** improve specialisation.
- A **linear recurrence** $h_t = a_t h_{t-1} + b_t$ is an associative scan: parallel $O(\log T)$-depth training, constant-size state at inference.
- **Selective SSMs (Mamba)** make $\Delta, B, C$ depend on the input, which is exactly what content-dependent tasks like selective copying need.
- Attention's superpower is **exact recall**; SSMs' superpower is **constant memory**. **Hybrids** take the best of both.

## 🎤 Explain it to a friend
Explain to a friend two tricks for making AI models cheaper. First, a **hive of specialists**: instead of every bee doing every job, a dispatcher sends each task to the two best-suited bees, and you have to stop the dispatcher from sending everything to the same favourite bee. Second, a **notebook instead of a photographic memory**: instead of re-reading the whole conversation for every word, the model keeps a fixed-size notebook it updates as it goes, and it *chooses* what's worth writing down.

## 💼 Interview corner
1. **What is the Switch Transformer load-balancing loss, and why does it multiply f_i by P_i?**
   <details><summary>Answer</summary>L = α·E·Σ_i f_i·P_i, where f_i is the fraction of tokens dispatched to expert i and P_i is the mean router probability for expert i. f_i comes from an argmax/top-k and has no gradient; P_i is differentiable. Their product is minimised (value 1 before α) by uniform routing, and its gradient pushes router probability away from experts that are currently overloaded (large f_i).</details>
2. **Why can a linear recurrence like an SSM be trained in parallel over time, while a classic LSTM can't (easily)?**
   <details><summary>Answer</summary>h_t = a_t·h_{t−1} + b_t is linear in h, so each step is an affine map and composing affine maps is associative: (a₁, b₁)•(a₂, b₂) = (a₁a₂, a₂b₁ + b₂). A parallel prefix scan then computes all h_t in O(log T) depth. An LSTM's state update passes h through nonlinearities (sigmoid/tanh of a matrix product), so steps don't compose into a fixed-size associative form.</details>
3. **When would you choose a hybrid (SSM + some attention) over a pure Transformer or a pure SSM?**
   <details><summary>Answer</summary>When you need long contexts or high-throughput generation (the KV cache of pure attention grows linearly with context and dominates memory) but still need precise retrieval of specific earlier tokens, which fixed-size recurrent states do poorly (e.g. associative recall). A few attention layers restore exact recall, while most layers stay linear-time with constant-size state.</details>

## 🐝 Fun fact
Mixture-of-Experts predates the Transformer era of LLMs: Shazeer et al.'s 2017 sparsely-gated MoE layer sat between stacked **LSTM** layers, and their largest model had about **137 billion parameters**, years before dense models reached that size.

## 📚 Further reading
- Shazeer et al., *Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer* (2017).
- Lepikhin et al., *GShard: Scaling Giant Models with Conditional Computation and Automatic Sharding* (2021).
- Fedus, Zoph & Shazeer, *Switch Transformers: Scaling to Trillion Parameter Models with Simple and Efficient Sparsity* (JMLR 2022).
- Zoph et al., *ST-MoE: Designing Stable and Transferable Sparse Expert Models* (2022).
- Jiang et al., *Mixtral of Experts* (2024).
- Dai et al., *DeepSeekMoE: Towards Ultimate Expert Specialization in Mixture-of-Experts Language Models* (2024).
- Wang et al., *Auxiliary-Loss-Free Load Balancing Strategy for Mixture-of-Experts* (2024); DeepSeek-AI, *DeepSeek-V3 Technical Report* (2024).
- Gu, Goel & Ré, *Efficiently Modeling Long Sequences with Structured State Spaces* (S4, ICLR 2022).
- Gu & Dao, *Mamba: Linear-Time Sequence Modeling with Selective State Spaces* (2023).
- Dao & Gu, *Transformers are SSMs: Generalized Models and Efficient Algorithms Through Structured State Space Duality* (Mamba-2, ICML 2024).
- Blelloch, *Prefix Sums and Their Applications* (1990); Martin & Cundy, *Parallelizing Linear Recurrent Neural Nets Over Sequence Length* (ICLR 2018).
- Katharopoulos et al., *Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention* (ICML 2020).
- Arora et al., *Zoology: Measuring and Improving Recall in Efficient Language Models* (2023).
- Yang, Kautz & Hatamizadeh, *Gated Delta Networks: Improving Mamba2 with Delta Rule* (ICLR 2025).
- Lieber et al., *Jamba: A Hybrid Transformer-Mamba Language Model* (2024).
- MiniMax, *MiniMax-01: Scaling Foundation Models with Lightning Attention* (2025).
- Sasha Rush & Sidd Karamcheti, *The Annotated S4* (blog); Maarten Grootendorst, *A Visual Guide to Mamba and State Space Models* (blog).